# 07 - Explainable AI

Feature importance and per-prediction explanations.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 50)

In [2]:
from app.ml.predict import get_engine
from app.explainability.feature_importance import (
    model_feature_importance, explain_prediction, warning_indicators,
)
from app.feature_engineering.extractor import FEATURE_NAMES, extract_features

engine = get_engine()
if not engine.is_ready():
    raise SystemExit("Train the models first: python -m app.ml.train")
model = engine.get_model('random_forest')
importances = model_feature_importance(model, FEATURE_NAMES)
pd.Series(importances).head(15)

has_https                   0.096468
suspicious_keyword_count    0.070590
is_suspicious_tld           0.061831
num_dots                    0.060810
has_digit_in_domain         0.047002
domain_digit_count          0.044674
hostname_entropy            0.034776
has_suspicious_keyword      0.028409
hostname_length             0.027680
url_length                  0.024837
url_entropy                 0.024008
digit_ratio                 0.023151
num_tokens                  0.022646
num_digits                  0.021631
digit_letter_ratio          0.021615
dtype: float64

## Explain one prediction

In [3]:
url = 'http://secure-login.paypal-verify9.xyz/account/confirm.php'
result = engine.predict_url(url, 'random_forest')
print(f"{url}\nPrediction: {result['prediction'].upper()}  Risk: {result['risk_score']:.2%}")

http://secure-login.paypal-verify9.xyz/account/confirm.php
Prediction: PHISHING  Risk: 95.67%


In [4]:
features = result['features']
for factor in explain_prediction(features, importances):
    print(f"  {factor['label']:<26} {factor['level']:<8} weight={factor['importance']:.3f}")

  HTTPS                      ABSENT   weight=0.096
  Suspicious keywords        HIGH     weight=0.071
  Top level domain           UNCOMMON weight=0.062
  URL length                 MEDIUM   weight=0.025
  Character entropy          HIGH     weight=0.024
  Digit count                LOW      weight=0.022


In [5]:
print('Warning indicators:')
for note in warning_indicators(features):
    print('  -', note)

Warning indicators:
  - Contains sensitive-action keywords
  - No HTTPS
  - High character entropy
  - Uncommon top level domain


## SHAP summary (optional)

In [6]:
from app.explainability.shap_analysis import SHAP_AVAILABLE, global_feature_importance
from app.ml.preprocessing import get_processed_dataset, split_features_labels

if SHAP_AVAILABLE:
    frame = get_processed_dataset()
    features_matrix, _ = split_features_labels(frame, FEATURE_NAMES)
    shap_importance = global_feature_importance(model, features_matrix, FEATURE_NAMES)
    print(pd.Series(shap_importance).head(15))
else:
    print('SHAP is not installed; falling back to the importance shown above.')

2026-09-26 13:15:29 | WARNING  | shap | Background dataset has 300 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=300 when initializing the masker.


/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 10%|==                  | 59/600 [00:11<01:40]       

 11%|==                  | 65/600 [00:12<01:38]       

 12%|==                  | 71/600 [00:13<01:36]       

...

 97%|=================== | 580/600 [01:40<00:03]       

 98%|===================| 586/600 [01:41<00:02]       

 99%|===================| 592/600 [01:42<00:01]       

100%|===================| 598/600 [01:43<00:00]       

has_https                   0.133401
is_suspicious_tld           0.085661
suspicious_keyword_count    0.070956
has_digit_in_domain         0.060891
domain_digit_count          0.060690
num_dots                    0.060039
has_suspicious_keyword      0.035152
keyword_in_path             0.033982
is_common_tld               0.033867
hostname_entropy            0.025678
hostname_length             0.024380
num_host_labels             0.019750
digit_letter_ratio          0.019590
digit_ratio                 0.019391
num_digits                  0.018916
dtype: float64
